### Прогнозирование PM2.5 для Delhi

Датасет: [Air Quality Data in India](https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india).

Цель: обучить регрессионную модель и сохранить **Pipeline** (импутация + масштабирование + Ridge) для синхронного инференса.

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pickle import dump
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from model_utils import FEATURE_COLUMNS

CITY = 'Delhi'
TARGET = 'PM2.5'

In [ ]:
df = pd.read_csv(ROOT / 'data' / 'city_day.csv', parse_dates=['Date'])
delhi = df[df['City'] == CITY].dropna(subset=[TARGET])
print(f'Строк для обучения: {len(delhi)}')
delhi[FEATURE_COLUMNS + [TARGET]].describe().T

In [ ]:
corr = delhi[FEATURE_COLUMNS + [TARGET]].corr()[TARGET].sort_values(ascending=False)
corr

In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_transformer, FEATURE_COLUMNS),
])
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge()),
])

In [ ]:
x = delhi[FEATURE_COLUMNS]
y = delhi[TARGET]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

search_cv = GridSearchCV(pipe, {'regressor__alpha': np.linspace(0, 10, 41)}, cv=5)
search_cv.fit(x_train, y_train)

y_pred = search_cv.predict(x_test)
print('Best params:', search_cv.best_params_)
print('Test R2:', r2_score(y_test, y_pred).round(3))
print('Test MAE:', round(mean_absolute_error(y_test, y_pred), 2), 'µg/m³')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.5)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
ax.set_xlabel('Факт PM2.5')
ax.set_ylabel('Прогноз PM2.5')
ax.set_title('Delhi: тестовая выборка')
plt.show()

In [ ]:
model_dir = ROOT / 'models'
model_dir.mkdir(parents=True, exist_ok=True)
with open(model_dir / 'pipeline.pkl', 'wb') as file:
    dump(search_cv, file)
print('Модель сохранена в models/pipeline.pkl')